# AIFM Real Estate Fund

This notebook presents a real estate risk-monitoring workflow for a simulated closed-ended mixed AIF portfolio. The fund combines direct property investments across European office, logistics, retail, and residential assets with listed REITs, an FX hedge, and cash.

The analysis keeps the sleeves explicitly separate: direct properties are monitored through appraised value, LTV, rental yield, vacancy, tenant risk, and property stress; listed REITs and FX use position-style market analytics. Because the fund is closed-ended, there is no redemption or asset-liquidity monitoring in this notebook.

> **Output gallery:** All tables and plots generated by this notebook are saved in the [fig/AIFM_RealEstate](../../fig/AIFM_RealEstate) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

In [ ]:
import warnings

from fund_risk_workflow.data.setup_db import run as setup_db
from fund_risk_workflow.data.mock_bloomberg import MockBloomberg as Bloomberg

import fund_risk_workflow.data.database as db
import fund_risk_workflow.risk.esg_utils as esg_u
import fund_risk_workflow.ui.print_html_utils as phtml
import fund_risk_workflow.ui.real_estate_display as red

warnings.filterwarnings("ignore")

setup_db()
ENGINE = db.get_engine()
BBG = Bloomberg()

## 1. Fund Setup and Risk Policy

### 1.1 Fund Example

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_RealEstate'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---

### 1.2 Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration. The property stress magnitudes, LTV thresholds, and the tenant-default capitalisation assumption are documented in the risk policy rather than in notebook code.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

### 1.3 Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# Fixed valuation date for all computations in this notebook
from fund_risk_workflow.config import VALUATION_DATE
VALUATION_DATE

The workflow builder loads positions from the SQLite data layer, enriches the listed sleeve through the simulated Bloomberg workflow, links the simulated lease register to the actual property ISINs, and computes every result used in this notebook. From this point onward, code cells contain only display calls.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# Build the full real estate monitoring result set
from fund_risk_workflow.pipeline.real_estate_workflow import build_real_estate_workflow

workflow = build_real_estate_workflow(
    engine=ENGINE,
    bbg=BBG,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
)
NAV = workflow["nav"]
rmp = workflow["rmp"]

---

## 2. Portfolio and Sleeve Overview

### 2.1 Portfolio Overview

The fund summary, largest positions, and asset-class breakdown describe the portfolio at the valuation date.

In [ ]:
phtml.display_fund_summary(FUND_ID, VALUATION_DATE, workflow["positions"], workflow["risk_df"], NAV, export_id="03")

In [ ]:
phtml.display_top_positions(workflow["risk_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(workflow["risk_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")

### 2.2 Sleeve Separation

The fund holds four sleeves with fundamentally different data sources and risk characteristics:

- **Direct properties**: quarterly appraisal valuation, no market ticker; monitored through LTV, rental yield, vacancy, and tenant risk.
- **Listed REITs**: daily market prices with Bloomberg enrichment.
- **FX hedge**: currency forward on the USD exposure.
- **Cash**: base-currency liquidity.

Direct properties are not treated as daily-traded securities anywhere in this notebook.

In [ ]:
red.display_sleeve_summary(workflow["sleeve_summary"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

---

## 3. Direct Property Monitoring

Key metrics for the direct property sleeve, sourced from the fund administrator quarterly valuation inputs:

- **LTV**: property-level debt as % of property value.
- **Rental yield**: annual contracted rent / property value, gross of vacancy.
- **Vacancy rate**: % of lettable space not generating income.
- **Effective yield**: rental yield × (1 − vacancy rate), the actual income yield.

Weighted averages are value-weighted across the four properties.

In [ ]:
red.display_direct_property_profile(workflow["direct_property_profile"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="07")

In [ ]:
ltv_warning_pct = rmp["ltv_monitoring"]["ltv_warning_threshold"] * 100
red.plot_direct_property_metrics(
    workflow["direct_property_profile"],
    FUND_ID,
    ltv_warning_pct=ltv_warning_pct,
    valuation_date=VALUATION_DATE,
    export_id="08",
)

---

## 4. Leverage

Fund-level leverage is computed with the same canonical Gross and Commitment method implementation used by the other AIF workflows. For this fund:

- The **gross method** captures the full portfolio exposure (direct properties, listed REITs, and the FX forward), excluding cash.
- Under the **commitment method**, the equity-style and bond-style buckets are empty and the hedged FX exposure nets out, so commitment exposure reflects only the unhedged FX notional. Property-level debt is ring-fenced at asset level and monitored through LTV rather than fund-level leverage.

The AIFMD II granular breakdown below classifies gross exposure by source and listed/OTC status.

In [ ]:
phtml.display_granular(workflow["granular_leverage"], NAV, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="09")

---

## 5. Property and Listed-Sleeve Stress

### 5.1 Scenario Assumptions

Stress assumptions are documented in the fund's risk policy (migrated from earlier notebook assumptions, unchanged). Property-type shocks and the rental stress apply to direct properties only; the rate shock and historical market scenarios apply only to the listed sleeve.

> Note on rates: the listed sleeve holds no duration-sensitive instruments, so the +200bps parallel shift shows no P&L. For direct properties the relevant transmission is cap-rate expansion, which is represented in the property value stress.

In [ ]:
red.display_stress_assumptions(workflow["stress_assumptions"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="10")

### 5.2 Scenario Results

Stress P&L is expressed against full fund NAV. Direct-property shocks revalue the appraised property values; historical market scenarios shock only the listed REIT and FX sleeve.

In [ ]:
red.display_stress_results(workflow["stress_results"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="11")

### 5.3 LTV Covenant Stress

The LTV stress applies the documented severe property value shock and tests each property's stressed LTV against the policy stress threshold. Stressed LTV = current LTV / (1 + value shock).

In [ ]:
red.display_ltv_stress(workflow["ltv_stress"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

---

## 6. Investor and Tenant Concentration

### 6.1 Investor Concentration

Investor concentration is monitored as a closed-ended ownership and governance indicator against ESMA thresholds (single investor 20% of NAV, top 3 investors 50%). The register is loaded from fund-level reference data and is simulated. Because the fund is closed-ended, concentration is not translated into a redemption scenario.

In [ ]:
red.display_investor_concentration_closed_ended(workflow["investor_concentration"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="13")

### 6.2 Tenant and Property Rental Concentration

Tenant risk is the primary income risk for the direct property sleeve. The lease register is simulated and linked to the actual property ISINs; property-level rent reconciles with the valuation inputs (value × rental yield × occupancy) within 1%. Tables show the register as-of date.

In [ ]:
red.display_tenant_concentration(
    workflow["tenant_concentration"],
    as_of_date=workflow["tenant_register"].attrs["as_of_date"],
    fund_id=FUND_ID,
    export_id="14",
)

### 6.3 Tenant Default Stress

The stress assumes the largest actual tenant exposure defaults with a one-year full void and no recovery. Lost income does not reduce NAV directly but impairs yield and asset value; the implied NAV impact capitalises the lost rent at the documented capitalisation yield. Register and assumptions are simulated.

In [ ]:
red.display_tenant_default_stress(workflow["tenant_default_stress"], fund_id=FUND_ID, export_id="15")

---

## 7. Sustainability Risk Indicators

Portfolio-level ESG indicators are calculated using NAV-weighted exposures: composite and pillar scores, low-score exposure versus the internal threshold, controversy flags, and carbon intensity.

ESG scores for listed REITs come from the market-data layer. Direct properties have no market ticker and use reference-data scores, which are simulated appraiser-style assessments.

> Scale note: ESG scores use a 0-100 scale, where 100 is best. ESG scores are sustainability-risk inputs and are not mapped to SFDR classifications here.

In [ ]:
esg_u.display_esg_assets(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="16")

In [ ]:
esg_u.display_esg_summary(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="17")

In [ ]:
esg_u.plot_esg_profile(workflow["esg_df"], FUND_ID, plot_title='ESG profile — Real Estate', valuation_date=VALUATION_DATE, export_id="18")

---

## 8. Annex IV Report

Selected outputs are mapped to Annex IV-style reporting fields. For this closed-ended fund the report covers identification, portfolio breakdown, and leverage detail; liquidity and redemption sections do not apply and are excluded. Investor and tenant concentration are monitored separately in Section 6 from the fund-level registers.

**Regulatory basis:** Delegated Regulation (EU) 231/2013 Article 110 and Annex IV reporting template.

In [ ]:
import fund_risk_workflow.reporting.annex_iv_workflow as annex_iv_workflow
from fund_risk_workflow.config import QUARTER

annex_iv_result = annex_iv_workflow.run(
    engine=ENGINE,
    fund_id=FUND_ID,
    quarter=QUARTER,
    first_export_id="19",
    sections=("identification", "breakdown", "leverage_detail"),
)